In [1]:
# Set the root path to your project directory
import pathlib

root = pathlib.Path(r"c:\Users\26293\Desktop\WORK\emhass_git\emhass")
print(root)
print(list((root / "data").glob("*")))  # List files in the data folder

c:\Users\26293\Desktop\WORK\emhass_git\emhass
[WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/actionLogs.txt'), WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/adjust_pv_regressor.pkl'), WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/cached-open-meteo-forecast.json'), WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/data_load_cost_forecast.csv'), WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/data_load_forecast.csv'), WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/data_prod_price_forecast.csv'), WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/data_weather_forecast.csv'), WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/debug-adjust-pv-forecast-data-prep-input-data.csv'), WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/debug-adjust-pv-forecast-data-prep-output-data.csv'), WindowsPath('c:/Users/26293/Desktop/WORK/emhass_git/emhass/data/debug-ad

In [2]:
import pickle

# Load the pickle file
with open(root / "data" / "long_train_data.pkl", "rb") as inp:
    data = pickle.load(inp)

# Check the type and length of the loaded object
print(type(data))
print(len(data))

df_final, days_list, var_list, ha_config = data

# Show the DataFrame
print("days_list:", days_list)
print("var_list:", var_list)
print("ha_config:", ha_config)

<class 'tuple'>
4
days_list: DatetimeIndex(['2024-02-28 21:00:00+00:00', '2024-02-29 21:00:00+00:00',
               '2024-03-01 21:00:00+00:00', '2024-03-02 21:00:00+00:00',
               '2024-03-03 21:00:00+00:00', '2024-03-04 21:00:00+00:00',
               '2024-03-05 21:00:00+00:00', '2024-03-06 21:00:00+00:00',
               '2024-03-07 21:00:00+00:00', '2024-03-08 21:00:00+00:00',
               ...
               '2025-02-18 21:00:00+00:00', '2025-02-19 21:00:00+00:00',
               '2025-02-20 21:00:00+00:00', '2025-02-21 21:00:00+00:00',
               '2025-02-22 21:00:00+00:00', '2025-02-23 21:00:00+00:00',
               '2025-02-24 21:00:00+00:00', '2025-02-25 21:00:00+00:00',
               '2025-02-26 21:00:00+00:00', '2025-02-27 21:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=366, freq='D')
var_list: ['sensor.power_load_no_var_loads', 'sensor.power_photovoltaics', 'sensor.p_pv_forecast']
ha_config: {'country': 'FR', 'currency': 'EUR', 'elevatio

C:\Users\26293\AppData\Local\Temp\ipykernel_36452\83218460.py:5: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(inp)


In [21]:
import pickle
import pathlib
import pandas as pd
from emhass.machine_learning_forecaster import MLForecaster
from emhass.forecast import Forecast
from emhass import utils

# Setup paths and logger
root = pathlib.Path(r"c:\Users\26293\Desktop\WORK\emhass_git\emhass")
emhass_conf = {
    "data_path": root / "data/",
    "root_path": root / "src/emhass/",
    "defaults_path": root / "src/emhass/data/config_defaults.json",
    "associations_path": root / "src/emhass/data/associations.csv",
}
logger, ch = utils.get_logger("notebook", emhass_conf, save_to_file=False)

# Load test data
with open(emhass_conf["data_path"] / "long_train_data.pkl", "rb") as inp:
    df_final, days_list, var_list, ha_config = pickle.load(inp)


In [6]:
# Example of using LightGBM
model_type = "long_train_data"
var_model = "sensor.power_load_no_var_loads"
sklearn_model = "LightGBM"  # Use the new model
num_lags = 48

mlf_lgb = MLForecaster(df_final, model_type, var_model, sklearn_model, num_lags, emhass_conf, logger)
print("==== MLForecaster Configuration ====")
print(f"Model type: {mlf_lgb.model_type}")
print(f"Target variable: {mlf_lgb.var_model}")
print(f"sklearn model: {mlf_lgb.sklearn_model}")
print(f"Number of lags: {mlf_lgb.num_lags}")
print("\n==== Data Info ====")
print(f"Data frequency: {mlf_lgb.data.index.freq}")
print(f"Data range: {mlf_lgb.data.index.min()} to {mlf_lgb.data.index.max()}")
print(f"Data points: {len(mlf_lgb.data)}")
df_pred_lgb, _ = mlf_lgb.fit()


2025-07-23 14:08:25,909 - notebook - INFO - Performing a forecast model fit for long_train_data
2025-07-23 14:08:25,915 - notebook - INFO - Training a LightGBM model
2025-07-23 14:08:26,057 - notebook - INFO - Elapsed time for model fit: 0.1427321434020996


==== MLForecaster Configuration ====
Model type: long_train_data
Target variable: sensor.power_load_no_var_loads
sklearn model: LightGBM
Number of lags: 48

==== Data Info ====
Data frequency: <30 * Minutes>
Data range: 2024-02-28 21:00:00+00:00 to 2025-02-27 22:30:00+00:00
Data points: 17524


2025-07-23 14:08:26,125 - notebook - INFO - Prediction R2 score of fitted model on test data: 0.7449937251889043
2025-07-23 14:08:26,129 - notebook - INFO - Model Evaluation Metrics:
2025-07-23 14:08:26,130 - notebook - INFO - R² Score: 0.7450
2025-07-23 14:08:26,130 - notebook - INFO - RMSE: 550.4279
2025-07-23 14:08:26,130 - notebook - INFO - MAE: 437.5288
2025-07-23 14:08:26,130 - notebook - INFO - MAPE: 56.36%


In [12]:
# Only keep rows where both test and pred are not NaN
test_comparison = df_pred_lgb[["test", "pred"]].dropna()
print(test_comparison.head())

# Print data ranges for better context
print("\n==== Data Ranges ====")
# Training data range (all data except test portion)
train_range_min = mlf_lgb.data.index.min()
train_range_max = test_comparison.index.min() - pd.Timedelta(minutes=30)  # Assuming 30-min intervals
print(f"Training data range: {train_range_min} to {train_range_max}")
# Test data range
test_range_min = test_comparison.index.min()
test_range_max = test_comparison.index.max()
print(f"Test data range: {test_range_min} to {test_range_max}")


import plotly.graph_objs as go
from plotly.offline import iplot

# Create interactive plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_comparison.index, y=test_comparison["test"], mode='lines', name='Actual (Test)'))
fig.add_trace(go.Scatter(x=test_comparison.index, y=test_comparison["pred"], mode='lines', name='Prediction'))

fig.update_layout(
    title="Test Set: Actual vs Prediction (Interactive)",
    xaxis_title="Time",
    yaxis_title="Value",
    legend=dict(x=0, y=1),
    hovermode="x unified" 
)

iplot(fig)

                                  test         pred
2025-02-25 23:00:00+00:00  3482.792056  1574.337029
2025-02-25 23:30:00+00:00  1557.221215  1120.557378
2025-02-26 00:00:00+00:00   914.423966  1288.112830
2025-02-26 00:30:00+00:00   266.457207  1218.645288
2025-02-26 01:00:00+00:00   232.014222  1192.135003

==== Data Ranges ====
Training data range: 2024-02-28 21:00:00+00:00 to 2025-02-25 22:30:00+00:00
Test data range: 2025-02-25 23:00:00+00:00 to 2025-02-27 22:30:00+00:00


In [14]:
df_pred, df_pred_backtest = mlf_lgb.fit(perform_backtest=True)
print(df_pred_backtest.tail())

2025-07-23 14:19:38,900 - notebook - INFO - Performing a forecast model fit for long_train_data
2025-07-23 14:19:38,907 - notebook - INFO - Training a LightGBM model
2025-07-23 14:19:39,038 - notebook - INFO - Elapsed time for model fit: 0.13075613975524902
2025-07-23 14:19:39,112 - notebook - INFO - Prediction R2 score of fitted model on test data: 0.7449937251889043
2025-07-23 14:19:39,116 - notebook - INFO - Model Evaluation Metrics:
2025-07-23 14:19:39,117 - notebook - INFO - R² Score: 0.7450
2025-07-23 14:19:39,117 - notebook - INFO - RMSE: 550.4279
2025-07-23 14:19:39,118 - notebook - INFO - MAE: 437.5288
2025-07-23 14:19:39,119 - notebook - INFO - MAPE: 56.36%
2025-07-23 14:19:39,125 - notebook - INFO - Performing simple backtesting of fitted model
100%|██████████| 363/363 [00:12<00:00, 28.02it/s]
2025-07-23 14:19:52,107 - notebook - INFO - Elapsed backtesting time: 12.979759931564331
2025-07-23 14:19:52,109 - notebook - INFO - Backtest R2 score:    neg_r2_score
0      0.502859


                                 train  pred
2025-02-27 20:30:00+00:00  3478.084000   NaN
2025-02-27 21:00:00+00:00  3409.921111   NaN
2025-02-27 21:30:00+00:00  3344.159392   NaN
2025-02-27 22:00:00+00:00  1563.476536   NaN
2025-02-27 22:30:00+00:00  1532.535000   NaN


In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
test_backtest_comparison = df_pred_backtest[["train", "pred"]].dropna()
test_backtest_comparison.tail()
r2 = r2_score(test_backtest_comparison["train"], test_backtest_comparison["pred"])
# Mean Absolute Error
mae = mean_absolute_error(test_backtest_comparison["train"], test_backtest_comparison["pred"])
# Root Mean Squared Error
rmse = np.sqrt(mean_squared_error(test_backtest_comparison["train"], test_backtest_comparison["pred"]))

# Display metrics
print("\nModel Performance Metrics:")
print(f"R² Score: {r2:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"RMSE (sensitive to large errors): {rmse:.4f}")


Model Performance Metrics:
R² Score: 0.5029
MAE (Mean Absolute Error): 479.4663
RMSE (sensitive to large errors): 691.2475


In [22]:
# # First fit with larger tuning window
df_pred, _ = mlf_lgb.fit(tuning_days="30days")  

# Then tune using that window
df_pred_opt = mlf_lgb.tune()
df_pred_opt.head()

2025-07-23 14:25:28,729 - notebook - INFO - Performing a forecast model fit for long_train_data
2025-07-23 14:25:28,729 - notebook - INFO - Performing a forecast model fit for long_train_data
2025-07-23 14:25:28,731 - notebook - INFO - Training a LightGBM model
2025-07-23 14:25:28,731 - notebook - INFO - Training a LightGBM model
2025-07-23 14:25:28,897 - notebook - INFO - Elapsed time for model fit: 0.16045188903808594
2025-07-23 14:25:28,897 - notebook - INFO - Elapsed time for model fit: 0.16045188903808594
2025-07-23 14:25:28,967 - notebook - INFO - Prediction R2 score of fitted model on test data: 0.7449937251889043
2025-07-23 14:25:28,967 - notebook - INFO - Prediction R2 score of fitted model on test data: 0.7449937251889043
2025-07-23 14:25:28,967 - notebook - INFO - Model Evaluation Metrics:
2025-07-23 14:25:28,967 - notebook - INFO - Model Evaluation Metrics:
2025-07-23 14:25:28,967 - notebook - INFO - R² Score: 0.7450
2025-07-23 14:25:28,967 - notebook - INFO - R² Score: 0.7

`Forecaster` refitted using the best-found lags and parameters, and the whole data set: 
  Lags: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72] 
  Parameters: {'n_estimators': 172, 'learning_rate': 0.07239168088128368, 'max_depth': 7, 'num_leaves': 48}
  Backtesting metric: -0.4177806314477924


,train,test,pred_optim
2024-02-28 21:00:00+00:00,0.000000,NaN,NaN
2024-02-28 21:30:00+00:00,0.000000,NaN,NaN
2024-02-28 22:00:00+00:00,27.710482,NaN,NaN
2024-02-28 22:30:00+00:00,55.420963,NaN,NaN
2024-02-28 23:00:00+00:00,83.131445,NaN,NaN


In [24]:
df_pred_opt_comparsion = df_pred_opt[["test", "pred_optim"]].dropna()
df_pred_opt_comparsion.tail()
print("\n==== Tuning Data Ranges ====")
tune_range = df_pred_opt[df_pred_opt["pred_optim"].notna()]
tune_range_min = tune_range.index.min()
tune_range_max = tune_range.index.max()
print(f"Tuning data range: {tune_range_min} to {tune_range_max}")

r2 = r2_score(df_pred_opt_comparsion["test"], df_pred_opt_comparsion["pred_optim"])
# Mean Absolute Error
mae = mean_absolute_error(df_pred_opt_comparsion["test"], df_pred_opt_comparsion["pred_optim"])
# Root Mean Squared Error
rmse = np.sqrt(mean_squared_error(df_pred_opt_comparsion["test"], df_pred_opt_comparsion["pred_optim"]))

# Display metrics
print("\nModel Performance Metrics:")
print(f"R² Score: {r2:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"RMSE (sensitive to large errors): {rmse:.4f}")


==== Tuning Data Ranges ====
Tuning data range: 2025-02-25 23:00:00+00:00 to 2025-02-26 22:30:00+00:00

Model Performance Metrics:
R² Score: 0.6398
MAE (Mean Absolute Error): 519.5581
RMSE (sensitive to large errors): 675.2624


In [25]:
import plotly.graph_objs as go

# Prepare comparison DataFrames
test_comparison = df_pred_lgb[['test', 'pred']].dropna()
tune_comparison = df_pred_opt[['test', 'pred_optim']].dropna()

# Align indices (if needed)
common_idx = test_comparison.index.intersection(tune_comparison.index)
test_comparison = test_comparison.loc[common_idx]
tune_comparison = tune_comparison.loc[common_idx]

# Plot
fig = go.Figure()
fig.add_trace(go.Scatter(x=common_idx, y=test_comparison['test'], mode='lines', name='Actual (Test)'))
fig.add_trace(go.Scatter(x=common_idx, y=test_comparison['pred'], mode='lines', name='Prediction (Fit)'))
fig.add_trace(go.Scatter(x=common_idx, y=tune_comparison['pred_optim'], mode='lines', name='Prediction (Tune)'))

fig.update_layout(
    title="Test Set: Actual vs Fit vs Tune Prediction",
    xaxis_title="Time",
    yaxis_title="Value",
    legend=dict(x=0, y=1),
    hovermode="x unified"
)
fig.show()